In [0]:
from pyspark.sql.functions import (
    col, sum as _sum, count, round as _round, 
    date_trunc, date_format, avg, countDistinct)
import pyspark.sql.functions as F
catalog = "shopsphere_catalog"
schema = "retail"

In [0]:
orders = spark.table(f"{catalog}.{schema}.orders")
order_items = spark.table(f"{catalog}.{schema}.order_items")

In [0]:
monthly_gmv = (
    orders
    .filter(col("order_status") == "DELIVERED")
    .join(order_items, "order_id")
    .withColumn("order_month", date_trunc("month", col("order_purchase_timestamp")))
    .groupBy("order_month")
    .agg(_round(_sum("price"), 2).alias("gmv_brl"), _round(_sum("freight_value"), 2).alias("total_freight"), count("order_id").alias("order_count"),
countDistinct("customer_id").alias("unique_customers"), _round(avg("price"), 2).alias("avg_order_value")) .orderBy("order_month"))
display(monthly_gmv)


In [0]:
monthly_gmv.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.marketing.monthly_gmv_summary")
print(f"monthly_gmv_summary saved: {monthly_gmv.count()} months")